
# 🌐 NLP

Ce notebook suit un pipeline complet de NLP, basé sur le plan retenu :

1. Text Acquisition  
2. Exploration et Analyse de Texte (EDA)   
3. Keywords extraction


---
## 1. Text Acquisition Chanson Formidable de Stromae

Nous allons collecter du texte à partir de différentes sources :

| Source | Outil utilisé | Exemple       | Caractéristique               |
| ------ | ------------- | ------------- | ----------------------------- |
| PDF    | pdfplumber    | Article arXiv | Texte scientifique, structuré |
| Audio  | Whisper       | Sermon 30 sec | Transcription orale           |

- Collecte de textes à partir de sites web, réseaux sociaux, documents, bases de données.  
- Notion de corpus : taille, diversité, domaine.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# Télécharger le fichier CSV directement depuis GitHub
!wget https://raw.githubusercontent.com/bidoscar/AFRICITIZEN-ACDS-Coding/main/Seance_9_NLP/Datasets/Paroles_de_formidable_Stromae.pdf

In [ ]:
# Télécharger le fichier CSV directement depuis GitHub
!wget https://raw.githubusercontent.com/bidoscar/AFRICITIZEN-ACDS-Coding/main/Seance_9_NLP/Datasets/Stromae_-_Formidable_FeelMP3.com.mp3

### 📄 Exemple PDF (extraction de texte)

In [ ]:
#!pip install pdfplumber

In [ ]:
import pdfplumber, os

pdf_path = "Paroles_de_formidable_Stromae.pdf"
text_pdf = ""

with pdfplumber.open(pdf_path) as pdf:

    for page in pdf.pages:
        text_pdf += page.extract_text()

In [ ]:
text_pdf

### 🎙 Exemple Audio → Texte (Whisper)

In [ ]:
#!pip install openai-whisper

In [ ]:
import whisper
model = whisper.load_model("tiny")
audio_path = "Stromae_-_Formidable_FeelMP3.com.mp3"
result = model.transcribe(audio_path)
text_audio = result.get("text","")

In [ ]:
print(text_audio)

## 2. WORD TOKENNIZATION

In [ ]:
import nltk

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    # Download the 'punkt' tokenizer models for all languages, including French.
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    # Download the 'punkt_tab' tokenizer models.
    nltk.download('punkt_tab')


try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
import string

In [ ]:
tokenized_text = word_tokenize(text_pdf)
print(tokenized_text)

In [ ]:
len(tokenized_text)

## **Normalisation**

In [ ]:
# garder uniquement les tokens alphabetiques
tokenized_text_alpha = [word for word in tokenized_text if word.isalpha()]

print(tokenized_text_alpha)

In [ ]:
len(tokenized_text_alpha)

In [ ]:
tokenized_text_alpha_lower = [word.lower() for word in tokenized_text_alpha]
print(tokenized_text_alpha_lower)

## **Stop words**

In [ ]:
stop_words = set(stopwords.words('french'))
stop_words

In [ ]:
tokenized_text_alpha_lower_no_stopwords = [word for word in tokenized_text_alpha_lower if word not in stop_words]
print(len(tokenized_text_alpha_lower_no_stopwords))
print(tokenized_text_alpha_lower_no_stopwords)

In [ ]:
my_stop_words = ["paroles", "couplet", "refrain"]
tokenized_text_alpha_lower_no_stopwords_perso = [word for word in tokenized_text_alpha_lower_no_stopwords if word not in my_stop_words]
print(len(tokenized_text_alpha_lower_no_stopwords_perso))
print(tokenized_text_alpha_lower_no_stopwords_perso)

## **Lemmatisation and stemmasation**

## 🔹 **Lemmatization**

* **But** : ramener un mot à sa **forme canonique** (lemmatisation) en respectant la grammaire et le dictionnaire.
* Exemple :

  * *"mangeons"* → **"manger"**
  * *"meilleures"* → **"meilleur"**
  * *"couraient"* → **"courir"**
* C’est **linguistiquement correct** → on garde le sens du mot.
* Outils : WordNet (anglais), spaCy/Stanza (français).

---

## 🔹 **Stemming**

* **But** : couper le mot pour obtenir une **racine** (radical) via des règles simples (pas de grammaire).
* Exemple :

  * *"mangeons"* → **"mange"**
  * *"meilleures"* → **"meilleur"** (ok ici mais pas toujours)
  * *"couraient"* → **"cour"** (perd le sens exact)
* C’est **plus rapide**, mais **plus brutal** → on risque de perdre de la précision.
* Outils : PorterStemmer, SnowballStemmer.

---

## ⚖️ Résumé rapide

| Méthode        | Exemple mot | Résultat   | Avantage                         | Inconvénient                              |
| -------------- | ----------- | ---------- | -------------------------------- | ----------------------------------------- |
| **Lemmatizer** | *couraient* | **courir** | Respecte le sens et la grammaire | Plus lent                                 |
| **Stemmer**    | *couraient* | **cour**   | Rapide, simple                   | Perte d’info, pas toujours un mot correct |

In [ ]:
lemmatizer = WordNetLemmatizer()
list_lemmas = [lemmatizer.lemmatize(w) for w in tokenized_text_alpha_lower_no_stopwords_perso]

In [ ]:
list_lemmas[26], list_lemmas[30], list_lemmas[6]

In [ ]:
tokenized_text_alpha_lower_no_stopwords_perso[26], tokenized_text_alpha_lower_no_stopwords_perso[30], tokenized_text_alpha_lower_no_stopwords_perso[6]

In [ ]:
stemmer = PorterStemmer()
list_stems = [stemmer.stem(w) for w in tokenized_text_alpha_lower_no_stopwords_perso]

In [ ]:
list_stems[26], list_stems[30], list_stems[6]


---
## 2. Exploration et Analyse de Texte (EDA)

- Statistiques de base : nombre de documents, nombre moyen de mots par document.  
- Vocabulaire unique : taille du lexique.  
- Fréquences des mots : top 10, distribution des longueurs de mots.  
- Visualisations : nuages de mots (wordcloud), histogrammes.  
- Importance de "voir" les données avant de les transformer.


In [ ]:
import re
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from collections import Counter

In [ ]:
def eda_report (text, top_words):
  vocab = set(text)
  counts = Counter(text)
  print(f"Taille du corpus : {len(text)} tokens")
  print(f"Taille du vocabulaire : {len(vocab)} tokens")
  print(f"Nombre moyen de tokens par document : {round(len(text)/len(vocab),2)}")
  print("Top 10 des tokens les plus fréquents :")
  for word in counts.most_common(top_words):
    print("------------")
    print(f"{word[0]} : {word[1]}")

In [ ]:
eda_report(tokenized_text_alpha_lower_no_stopwords_perso, 30)

In [ ]:
# Histogramme des longueurs de mots dans le corpus
word_lengths = [len(word) for word in tokenized_text_alpha_lower_no_stopwords_perso]
plt.hist(word_lengths, bins=20, edgecolor='black')
plt.xlabel('Longueur des mots')
plt.ylabel('Fréquence')
plt.title('Distribution des longueurs de mots')
plt.show()

In [ ]:
my_vocab = set(tokenized_text_alpha_lower_no_stopwords_perso)

In [ ]:
# Histogramme des longueurs de mots dans le vocabulaire
word_lengths = [len(word) for word in my_vocab]
plt.hist(word_lengths, bins=20, edgecolor='black')
plt.xlabel('Longueur des mots')
plt.ylabel('Fréquence')
plt.title('Distribution des longueurs de mots')
plt.show()

In [ ]:
# Nuage des mots
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(tokenized_text_alpha_lower_no_stopwords_perso))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Nuage de mots')
plt.show()

# **Modelisation**

In [ ]:
#!pip install spacy scikit-learn yake sumy

In [ ]:
!python -m spacy download fr_core_news_sm

In [ ]:
import spacy

# Charger le modèle français
try:
    nlp = spacy.load("fr_core_news_sm")
except OSError:
    raise RuntimeError("Modèle spaCy 'fr_core_news_sm' manquant. Lance: python -m spacy download fr_core_news_sm")

## **Topic Modelling**

In [ ]:
#!pip install gensim

In [ ]:
#!pip install pyLDAvis

In [ ]:
from gensim import corpora, models
from gensim.models import CoherenceModel
import matplotlib.pyplot as plt

# Exemple : tes données segmentées en plusieurs sous-documents
texts = segments  # ex. liste de listes de tokens (chaque segment = 1 document)

# 1️⃣ Créer le dictionnaire et le corpus
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

# 2️⃣ Fonction pour calculer la cohérence pour plusieurs valeurs de k
def compute_coherence_values(dictionary, corpus, texts, start, limit, step):
    coherence_values = []
    model_list = []
    for num_topics in range(start, limit, step):
        model = models.LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            random_state=42,
            passes=10,
            alpha='auto'
        )
        model_list.append(model)
        coherencemodel = CoherenceModel(
            model=model,
            texts=texts,
            dictionary=dictionary,
            coherence='c_v'  # mesure basée sur similarité sémantique
        )
        coherence_values.append(coherencemodel.get_coherence())
    return model_list, coherence_values

# 3️⃣ Calculer la cohérence pour k=2 à 10
model_list, coherence_values = compute_coherence_values(dictionary, corpus, texts, start=2, limit=11, step=1)

# 4️⃣ Visualiser la cohérence
x = range(2, 11)
plt.figure(figsize=(8, 5))
plt.plot(x, coherence_values, marker='o')
plt.title("Détermination du nombre optimal de topics (Coherence Score)")
plt.xlabel("Nombre de topics")
plt.ylabel("Score de cohérence")
plt.grid(True)
plt.show()

# 5️⃣ Trouver la meilleure valeur de k
best_k = x[coherence_values.index(max(coherence_values))]
print(f"Nombre optimal de topics : {best_k}")


## **Resume**